In [1]:
import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display

## Load Project Data File

In [2]:
url = "https://raw.githubusercontent.com/OSU-fifem/OSU-ESE450/main/project/Yachats_Off_Grid_2023.csv"
df = pd.read_csv(url)

## Plot Data Interactively

In [ ]:
output_df = df.copy()

plot_columns = [
    'Temperature', 'Wind_Speed', 'Wind_Direction', 'DNI', 'DHI', 'GHI',
    'Streamflow_m3_s',
    'Base_Load_kW', 'Lighting_Plugs_kW',
    'Induction_Range_kW', 'Total_Electrical_Load_kW',
    'DHW_Demand_Liters'
]

df_plot = output_df.copy()
df_plot['Timestamp'] = pd.date_range(start="2023-01-01 00:00:00", periods=len(df_plot), freq="h")
df_plot.set_index('Timestamp', inplace=True)

range_selector = [
    dict(count=1, label="1d", step="day", stepmode="backward"),
    dict(count=7, label="7d", step="day", stepmode="backward"),
    dict(count=1, label="1m", step="month", stepmode="backward"),
    dict(count=3, label="3m", step="month", stepmode="backward"),
    dict(step="all")
]

def update_plot(variable, start_date, end_date, daily_mean):
    start_date = pd.to_datetime(start_date)
    end_date = pd.to_datetime(end_date)
    if start_date > end_date:
        start_date, end_date = end_date, start_date

    data = df_plot.loc[start_date:end_date, [variable]]
    if daily_mean:
        data = data.resample("D").mean()
        mode_label = "Daily average"
    else:
        mode_label = "Hourly"

    data = data.reset_index()
    fig = px.line(
        data,
        x='Timestamp',
        y=variable,
        title=f"{variable} ({mode_label})",
        labels={variable: variable, "Timestamp": "Date"}
    )
    fig.update_layout(
        xaxis=dict(
            rangeslider=dict(visible=True),
            rangeselector=dict(buttons=range_selector)
        ),
        margin=dict(l=40, r=20, t=50, b=30)
    )
    fig.show()

variable_picker = widgets.Dropdown(
    options=plot_columns,
    value='Total_Electrical_Load_kW',
    description="Metric:"
)

start_picker = widgets.DatePicker(
    value=pd.to_datetime("2023-01-01"),
    description="Start"
)

end_picker = widgets.DatePicker(
    value=pd.to_datetime("2023-12-31"),
    description="End"
)

daily_checkbox = widgets.Checkbox(
    value=False,
    description="Daily average"
)

controls = widgets.VBox([
    widgets.HBox([variable_picker, daily_checkbox]),
    widgets.HBox([start_picker, end_picker])
])

output = widgets.interactive_output(
    update_plot,
    {
        "variable": variable_picker,
        "start_date": start_picker,
        "end_date": end_picker,
        "daily_mean": daily_checkbox
    }
)

display(controls, output)

Output()